In [10]:
!pip  install sklearn-crfsuite

In [11]:
#Libraries
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
from tabulate import tabulate

Data Load

In [12]:
import os

localPath = "mypos-ver.3.0.shuf.nopipe.txt"

if os.path.exists(localPath):
    with open(localPath, "r", encoding="utf-8") as f:
        lines = f.readlines()

    print(f"Total sentences: {len(lines)}")
    print("\n--- First 15 Lines ---\n" + "\n".join(line.strip() for line in lines[:15]))
else:
    print(f"Error: '{localPath}' file was not found! Please check the file path or file name.")

Total sentences: 43196

--- First 15 Lines ---
၁၉၆၂/num ခုနှစ်/n ခန့်မှန်း/v သန်းခေါင်စာရင်း/n အရ/ppm လူဦးရေ/n ၁၁၅၉၃၁/num ယောက်/part ရှိ/v သည်/ppm ။/punc
လူ/n တိုင်း/part တွင်/ppm သင့်မြတ်/v လျော်ကန်/v စွာ/part ကန့်သတ်/v ထား/part သည့်/part အလုပ်/n လုပ်/v ချိန်/n အပြင်/conj ၊/punc လစာ/n နှင့်တကွ/conj အခါ/n ကာလ/n အားလျော်စွာ/ppm သတ်မှတ်/v ထား/part သည့်/part အလုပ်/n အားလပ်ရက်/n များ/part ပါဝင်/v သည့်/part အနားယူခွင့်/n နှင့်/conj အားလပ်ခွင့်/n ခံစားပိုင်ခွင့်/n ရှိ/v သည်/ppm ။/punc
ဤ/adj နည်း/n ကို/ppm စစ်ယူ/v သော/part နည်း/n ဟု/part ခေါ်/v သည်/ppm ။/punc
စာပြန်ပွဲ/n ဆို/v တာ/part က/ppm အာဂုံဆောင်/v အလွတ်ကျက်/v ထား/part တဲ့/part ပိဋကတ်သုံးပုံ/n စာပေ/n တွေ/part ကို/ppm စာစစ်/v သံဃာတော်ကြီး/n တွေ/part ရဲ့/ppm ရှေ့/n မှာ/ppm အလွတ်/adv ပြန်/v ပြီး/part ရွတ်ပြ/v ရ/part တာ/part ပေါ့/part ။/punc
ဒီ/pron မှာ/ppm ကျွန်တော့်/pron သက်သေခံကတ်/n ပါ/part ။/punc
၂ဝ/num ရာစု/n မြန်မာ့/n သမိုင်း/n သန်းဝင်းလှိုင်/n ၊/punc ၂ဝဝ၉/num ခု/part ၊/punc မေ/n လ/n ၊/punc ကံကော်ဝတ်ရည်/n စာပေ/n ။/punc
ကျွန်တော်/pron မ

Feature Extraction

In [13]:
def load_POSdata(path):
    with open(path, encoding='utf-8') as f:
        return [
            [(w, t) for item in line.split() if '/' in item for w, t in [item.rsplit('/', 1)]]
            for line in f if line.strip()
        ]

posdata = load_POSdata(localPath)
print(f"Total sentences: {len(posdata):,}")
print("sample sentence:", posdata[1])

Total sentences: 43,196
sample sentence: [('လူ', 'n'), ('တိုင်း', 'part'), ('တွင်', 'ppm'), ('သင့်မြတ်', 'v'), ('လျော်ကန်', 'v'), ('စွာ', 'part'), ('ကန့်သတ်', 'v'), ('ထား', 'part'), ('သည့်', 'part'), ('အလုပ်', 'n'), ('လုပ်', 'v'), ('ချိန်', 'n'), ('အပြင်', 'conj'), ('၊', 'punc'), ('လစာ', 'n'), ('နှင့်တကွ', 'conj'), ('အခါ', 'n'), ('ကာလ', 'n'), ('အားလျော်စွာ', 'ppm'), ('သတ်မှတ်', 'v'), ('ထား', 'part'), ('သည့်', 'part'), ('အလုပ်', 'n'), ('အားလပ်ရက်', 'n'), ('များ', 'part'), ('ပါဝင်', 'v'), ('သည့်', 'part'), ('အနားယူခွင့်', 'n'), ('နှင့်', 'conj'), ('အားလပ်ခွင့်', 'n'), ('ခံစားပိုင်ခွင့်', 'n'), ('ရှိ', 'v'), ('သည်', 'ppm'), ('။', 'punc')]


feature for CRF Model

In [14]:
def word2features(sent, i):
    word = sent[i][0]

    features = {
        'bias': 1.0,
        'word': word,
        'word.len': len(word),
        'word.prefix-1': word[:1],
        'word.prefix-2': word[:2],
        'word.suffix-1': word[-1:],
        'word.suffix-2': word[-2:],
        'word.is_digit': word.isdigit(),
    }

    # Previous word features
    if i > 0:
        word1 = sent[i-1][0]
        features.update({
            '-1:word': word1,
            '-1:word.len': len(word1),
            '-1:word.suffix-1': word1[-1:],
        })
    else:
        features['BOS'] = True  # Begin of Sentence

    # Next word features
    if i < len(sent) - 1:
        word1 = sent[i+1][0]
        features.update({
            '+1:word': word1,
            '+1:word.len': len(word1),
            '+1:word.suffix-1': word1[-1:],
        })
    else:
        features['EOS'] = True  # End of Sentence

    return features

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [label for token, label in sent]

In [15]:
sent2features(posdata[0])[3]

{'bias': 1.0,
 'word': 'သန်းခေါင်စာရင်း',
 'word.len': 15,
 'word.prefix-1': 'သ',
 'word.prefix-2': 'သန',
 'word.suffix-1': 'း',
 'word.suffix-2': '်း',
 'word.is_digit': False,
 '-1:word': 'ခန့်မှန်း',
 '-1:word.len': 9,
 '-1:word.suffix-1': 'း',
 '+1:word': 'အရ',
 '+1:word.len': 2,
 '+1:word.suffix-1': 'ရ'}

Training test split

In [17]:
from sklearn.model_selection import train_test_split

# Train-Test Split (0.2= 20% Test)
train_sents, test_sents = train_test_split(posdata, test_size=0.20, random_state=42)

X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

print(f"Training set : {len(X_train):,} sentences")
print(f"Testing set :  {len(X_test):,} sentences")

Training set : 34,556 sentences
Testing set :  8,640 sentences


Training CRF model

In [18]:
import sklearn_crfsuite

crfsuite = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True
)
crfsuite.fit(X_train, y_train)
print ("Model trainning is completed.")

Model trainning is completed.


Evaluation

In [19]:
y_pred = crfsuite.predict(X_test)
labels = list(crfsuite.classes_)
f1 = metrics.flat_f1_score(y_test, y_pred,
                      average='weighted', labels=labels)
accuarcy = metrics.flat_accuracy_score(y_test, y_pred)
print(f"=== Model Evaluation Results ===")
print(f"Accuracy : {accuarcy:.2%}")
print(f"F1 score : {f1:.2%}\n")

=== Model Evaluation Results ===
Accuracy : 96.03%
F1 score : 96.01%



In [20]:
print("Detailed classification report for test data")
print(metrics.flat_classification_report(
    y_test, y_pred, labels=labels, digits=2
))

Detailed classification report for test data
              precision    recall  f1-score   support

           n       0.96      0.97      0.96     24227
           v       0.95      0.94      0.94     16611
         ppm       0.98      0.98      0.98     17029
        punc       1.00      1.00      1.00     10782
         adj       0.85      0.80      0.82      3292
        part       0.97      0.97      0.97     26702
        conj       0.89      0.93      0.91      3474
         int       0.94      0.91      0.93       137
          fw       0.98      0.99      0.98       598
         num       1.00      1.00      1.00      1174
        pron       0.96      0.96      0.96      4021
         adv       0.91      0.83      0.87      2167
          tn       0.98      0.97      0.97      1172
          sb       1.00      0.81      0.90        58
         abb       0.95      0.75      0.84        69

    accuracy                           0.96    111513
   macro avg       0.95      0.92  

In [21]:
def pos_tag_sentence(words, model):
    dummy= [(w, 'UNK') for w in words]
    features = sent2features(dummy)
    tags = model.predict_single(features)
    return [(w, tag_descriptions.get(t, t)) for w, t in zip(words,tags)]

In [22]:
# POS Tag mean and dictionary!
tag_descriptions = {
    'n'   :   'နာမ်',
    'v'   :   'ကြိယာ',
    'ppm' :   'ဝိဘတ်',
    'part':   'ပစ္စည်း',
    'pron':   'နာမ်စား',
    'adj' :   'နာမဝိသေသန',
    'adv' :   'ကြိယာဝိသေသန',
    'punc':   'ပုဒ်ဖြတ်ပုဒ်ရပ်',
    'num' :   'ဂဏန်း',
    'conj':   'စကားဆက်'
}

In [23]:
sample_words = ["နည်းပညာ", "တိုးတက်", "လာသည်", "နှင့်အမျှ", "လူငယ်များ", "သည်", "ကွန်ပျူတာ", "ကို", "ကျွမ်းကျင်စွာ", "အသုံးပြု", "လာကြသည်", "။"]

results = pos_tag_sentence(sample_words, crfsuite)

tagged = [(word, tag_descriptions.get(tag, tag)) for word, tag in results]

print("\nSample Prediction Output:")
for word, tag in tagged:
    print(f"{word} ({tag})")


Sample Prediction Output:
နည်းပညာ (နာမ်)
တိုးတက် (ကြိယာ)
လာသည် (ပစ္စည်း)
နှင့်အမျှ (စကားဆက်)
လူငယ်များ (နာမ်)
သည် (ဝိဘတ်)
ကွန်ပျူတာ (နာမ်)
ကို (ဝိဘတ်)
ကျွမ်းကျင်စွာ (ကြိယာဝိသေသန)
အသုံးပြု (ကြိယာ)
လာကြသည် (ပစ္စည်း)
။ (ပုဒ်ဖြတ်ပုဒ်ရပ်)
